Ten skrypt implementuje system Retrieval-Augmented Generation (RAG) używający biblioteki CrewAI, który kieruje zapytania użytkownika do odpowiednich źródeł informacji poprzez zespół współpracujących agentów. System składa się z pięciu agentów. Całość tworzy pipeline, który przetwarza zapytanie użytkownika.

# Setup

In [ ]:
!uv pip install -q crewai crewai_tools langchain_community langchain_openai

In [3]:
from langchain_openai import ChatOpenAI
from langchain_community.tools.tavily_search import TavilySearchResults
from crewai.tools import tool
from crewai_tools import PDFSearchTool
from crewai import Crew, Task, Agent
import os
from google.colab import userdata

import requests

/usr/local/lib/python3.11/dist-packages/pydantic/_internal/_generate_schema.py:623: UserWarning: <built-in function callable> is not a Python type (it may be an instance of an object), Pydantic will allow any object with no validation since we cannot even enforce that the input is an instance of the given type. To get rid of this error wrap the type with `pydantic.SkipValidation`.
  warn(


In [ ]:
class CFG:
    model = "gpt-4o-mini"
    emb_model = "BAAI/bge-small-en-v1.5"
    temperature = 0.1
    max_tokens = 1000
    paper = "https://proceedings.neurips.cc/paper_files/paper/2017/file/3f5ee243547dee91fbd053c1c4a845aa-Paper.pdf"

In [ ]:
os.environ["OPENAI_API_KEY"] = userdata.get("openaivision")
os.environ["TAVILY_API_KEY"] = userdata.get("tavily")

# Model

In [ ]:
llm = ChatOpenAI(
    model_name=CFG.model, temperature=CFG.temperature, max_tokens=CFG.max_tokens
)

# Narzędzia

In [10]:
response = requests.get(CFG.paper)
with open("attenstion_is_all_you_need.pdf", "wb") as file:
    file.write(response.content)

In [ ]:
rag_tool = PDFSearchTool(
    pdf="attenstion_is_all_you_need.pdf",
    config=dict(
        llm=dict(
            provider="openai",
            config=dict(
                model=CFG.model,
                temperature=CFG.temperature,
            ),
        ),
        embedder=dict(
            provider="huggingface",
            config=dict(
                model=CFG.emb_model,
            ),
        ),
    ),
)

/usr/local/lib/python3.11/dist-packages/embedchain/embedder/huggingface.py:34: LangChainDeprecationWarning: The class `HuggingFaceEmbeddings` was deprecated in LangChain 0.2.2 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-huggingface package and should be used instead. To use it run `pip install -U :class:`~langchain-huggingface` and import as `from :class:`~langchain_huggingface import HuggingFaceEmbeddings``.
  embeddings = HuggingFaceEmbeddings(model_name=self.config.model, model_kwargs=self.config.model_kwargs)


In [13]:
# safety check

rag_tool.run("How did self-attention mechanism evolve in large language models?")

Using Tool: Search a PDF's content


/usr/local/lib/python3.11/dist-packages/chromadb/types.py:144: PydanticDeprecatedSince211: Accessing the 'model_fields' attribute on the instance is deprecated. Instead, you should access this attribute from the model class. Deprecated in Pydantic V2.11 to be removed in V3.0.
  return self.model_fields  # pydantic 2.x


'Relevant Content:\nmakes it more difﬁcult to learn dependencies between distant positions [ 11]. In the Transformer this is reduced to a constant number of operations, albeit at the cost of reduced effective resolution due to averaging attention-weighted positions, an effect we counteract with Multi-Head Attention as described in section 3.2. Self-attention, sometimes called intra-attention is an attention mechanism relating different positions of a single sequence in order to compute a representation of the sequence. Self-attention has been used successfully in a variety of tasks including reading comprehension, abstractive summarization, textual entailment and learning task-independent sentence representations [4, 22, 23, 19]. End-to-end memory networks are based on a recurrent attention mechanism instead of sequence- aligned recurrence and have been shown to perform well on simple-language question answering and language modeling tasks [28]. To the best of our knowledge, however, t

In [ ]:
web_search_tool = TavilySearchResults(max_results=3)
web_search_tool.run("What is self-attention mechansim in large language models?")

[{'title': 'Exploring Attention Mechanisms in Large Language Models - Medium',
  'url': 'https://medium.com/data-science-community-srm/exploring-attention-mechanisms-in-large-language-models-33549ae2c5b8',
  'content': 'Imagine each word in a sentence as a friend in a group chat. The self-attention mechanism lets the model look at each friend’s message, and based on what they said, decide how much attention to give to each one. This helps the model figure out the context and meaning of the words, making them really good at understanding language. [...] In the Transformer architecture, the self-attention mechanism plays a crucial role by helping the model decide how important each word or token is in a sequence. It does this by giving attention scores to words based on how relevant they are to others in the sequence. This way, the model can focus on the most contextually important information. [...] Now, self-attention is a feature in their design that helps them understand words and la

In [ ]:
@tool
def router_tool(question):
    """Router Function"""
    if "self-attention" in question:
        return "vectorstore"
    else:
        return "web_search"

Ten kod definiuje niestandardowe narzędzie o nazwie `router_tool`, które służy do kierowania zapytań do odpowiednich źródeł informacji.

*   `@tool`: Jest to dekorator, który oznacza funkcję jako narzędzie dostępne dla agentów AI w frameworku CrewAI.
*   `def router_tool(question):`: Definiuje funkcję o nazwie `router_tool`, która przyjmuje jeden argument: `question` (zapytanie).
*   `"""Router Function"""`: Jest to docstring, który opisuje przeznaczenie funkcji – jest to funkcja routingu.
*   `if 'self-attention' in question:`: Sprawdza, czy w zapytaniu występuje fraza "self-attention".
    *   `return 'vectorstore'`: Jeśli fraza "self-attention" znajduje się w zapytaniu, funkcja zwraca wartość `'vectorstore'`, co sugeruje, że zapytanie powinno być skierowane do wektora bazy danych (zawierającej informacje z pliku PDF).
*   `else:`: Jeśli fraza "self-attention" nie występuje w zapytaniu.
    *   `return 'web_search'`: Funkcja zwraca wartość `'web_search'`, co sugeruje, że zapytanie powinno być skierowane do wyszukiwarki internetowej.

W skrócie, funkcja `router_tool` analizuje treść zapytania i na podstawie obecności frazy "self-attention" decyduje, czy odpowiedź powinna być szukana w lokalnej bazie wektorowej (z pliku PDF), czy w internecie za pomocą wyszukiwarki.

# Agenci

In [ ]:
Router_Agent = Agent(
    role="Router",
    goal="Route user question to a vectorstore or web search",
    backstory=(
        "You are an expert at routing a user question to a vectorstore or web search."
        "Use the vectorstore for questions on concept related to Retrieval-Augmented Generation."
        "You do not need to be stringent with the keywords in the question related to these topics. Otherwise, use web-search."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
)

In [ ]:
Retriever_Agent = Agent(
    role="Retriever",
    goal="Use the information retrieved from the vectorstore to answer the question",
    backstory=(
        "You are an assistant for question-answering tasks."
        "Use the information present in the retrieved context to answer the question."
        "You have to provide a clear concise answer."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
)

In [ ]:
Grader_agent = Agent(
    role="Answer Grader",
    goal="Filter out erroneous retrievals",
    backstory=(
        "You are a grader assessing relevance of a retrieved document to a user question."
        "If the document contains keywords related to the user question, grade it as relevant."
        "It does not need to be a stringent test.You have to make sure that the answer is relevant to the question."
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
)

In [19]:
hallucination_grader = Agent(
    role="Hallucination Grader",
    goal="Filter out hallucination",
    backstory=(
        "You are a hallucination grader assessing whether an answer is grounded in / supported by a set of facts."
        "Make sure you meticulously review the answer and check if the response provided is in alignmnet with the question asked"
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
)

In [20]:
answer_grader = Agent(
    role="Answer Grader",
    goal="Filter out hallucination from the answer.",
    backstory=(
        "You are a grader assessing whether an answer is useful to resolve a question."
        "Make sure you meticulously review the answer and check if it makes sense for the question asked"
        "If the answer is relevant generate a clear and concise response."
        "If the answer gnerated is not relevant then perform a websearch using 'web_search_tool'"
    ),
    verbose=True,
    allow_delegation=False,
    llm=llm,
)

# Zadania

In [ ]:
router_task = Task(
    description=(
        "Analyse the keywords in the question {question}"
        "Based on the keywords decide whether it is eligible for a vectorstore search or a web search."
        "Return a single word 'vectorstore' if it is eligible for vectorstore search."
        "Return a single word 'websearch' if it is eligible for web search."
        "Do not provide any other preamble or explanation."
    ),
    expected_output=(
        "Give a binary choice 'websearch' or 'vectorstore' based on the question"
        "Do not provide any other preamble or explanation."
    ),
    agent=Router_Agent,
    tools=[router_tool],
)

In [ ]:
retriever_task = Task(
    description=(
        "Based on the response from the router task extract information for the question {question} with the help of the respective tool."
        "Use the web_serach_tool to retrieve information from the web in case the router task output is 'websearch'."
        "Use the rag_tool to retrieve information from the vectorstore in case the router task output is 'vectorstore'."
    ),
    expected_output=(
        "You should analyse the output of the 'router_task'"
        "If the response is 'websearch' then use the web_search_tool to retrieve information from the web."
        "If the response is 'vectorstore' then use the rag_tool to retrieve information from the vectorstore."
        "Return a claer and consise text as response."
    ),
    agent=Retriever_Agent,
    context=[router_task],
)

In [ ]:
grader_task = Task(
    description=(
        "Based on the response from the retriever task for the quetion {question} evaluate whether the retrieved content is relevant to the question."
    ),
    expected_output=(
        "Binary score 'yes' or 'no' score to indicate whether the document is relevant to the question"
        "You must answer 'yes' if the response from the 'retriever_task' is in alignment with the question asked."
        "You must answer 'no' if the response from the 'retriever_task' is not in alignment with the question asked."
        "Do not provide any preamble or explanations except for 'yes' or 'no'."
    ),
    agent=Grader_agent,
    context=[retriever_task],
)

In [ ]:
hallucination_task = Task(
    description=(
        "Based on the response from the grader task for the quetion {question} evaluate whether the answer is grounded in / supported by a set of facts."
    ),
    expected_output=(
        "Binary score 'yes' or 'no' score to indicate whether the answer is sync with the question asked"
        "Respond 'yes' if the answer is in useful and contains fact about the question asked."
        "Respond 'no' if the answer is not useful and does not contains fact about the question asked."
        "Do not provide any preamble or explanations except for 'yes' or 'no'."
    ),
    agent=hallucination_grader,
    context=[grader_task],
)

In [ ]:
answer_task = Task(
    description=(
        "Based on the response from the hallucination task for the quetion {question} evaluate whether the answer is useful to resolve the question."
        "If the answer is 'yes' return a clear and concise answer."
        "If the answer is 'no' then perform a 'websearch' and return the response"
    ),
    expected_output=(
        "Return a clear and concise response if the response from 'hallucination_task' is 'yes'."
        "Perform a web search using 'web_search_tool' and return ta clear and concise response only if the response from 'hallucination_task' is 'no'."
        "Otherwise respond as 'Sorry! unable to find a valid response'."
    ),
    context=[hallucination_task],
    agent=answer_grader,
)

# Załoga

In [ ]:
rag_crew = Crew(
    agents=[
        Router_Agent,
        Retriever_Agent,
        Grader_agent,
        hallucination_grader,
        answer_grader,
    ],
    tasks=[router_task, retriever_task, grader_task, hallucination_task, answer_task],
    verbose=True,
)

Ten kod tworzy zespół agentów (Crew) o nazwie `rag_crew`, który będzie współpracował w celu udzielenia odpowiedzi na pytanie użytkownika.

*   `rag_crew = Crew(...)`: Tworzy nowy obiekt o nazwie `rag_crew` będący instancją klasy `Crew`.
*   `agents=[Router_Agent, Retriever_Agent, Grader_agent, hallucination_grader, answer_grader]`: Określa listę agentów wchodzących w skład zespołu. Kolejność agentów na liście definiuje kolejność ich działania. Są to: agent routingu (`Router_Agent`), agent pobierający informacje (`Retriever_Agent`), agent oceniający trafność (`Grader_agent`), agent oceniający halucynacje (`hallucination_grader`) i agent formułujący odpowiedź (`answer_grader`).
*   `tasks=[router_task, retriever_task, grader_task, hallucination_task, answer_task]`: Określa listę zadań, które mają być wykonane przez zespół. Kolejność zadań na liście definiuje kolejność ich wykonywania. Każde zadanie jest przypisane do konkretnego agenta.
*   `verbose=True`: Włącza tryb szczegółowy, co oznacza, że będą wyświetlane informacje o działaniach zespołu i poszczególnych agentów.

W rezultacie powstaje zespół `rag_crew`, który jest gotowy do pracy nad udzieleniem odpowiedzi na pytanie użytkownika poprzez sekwencyjne wykonywanie zadań przez odpowiednich agentów. Zespół ten wykorzystuje technikę RAG (Retrieval-Augmented Generation) – pobiera informacje z różnych źródeł, ocenia ich trafność i generuje odpowiedź opartą na tych informacjach.

In [27]:
inputs = {"question": "Tell me about self-attention mechanism in Transformers?"}
result = rag_crew.kickoff(inputs=inputs)


╭──────────────────────────────────────────── Crew Execution Started ─────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Started                                                                                         │
│  Name: crew                                                                                                     │
│  ID: 14fb2359-4057-4646-b909-5000043623b0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

🚀 Crew: crew
└── 📋 Task: 6cb1dd88-792a-435a-84a6-58906208a4dc
       Status: Executing Task...

🚀 Crew: crew
└── 📋 Task: 6cb1dd88-792a-435a-84a6-58906208a4dc
       Status: Executing Task...
    └── 🤖 Agent: Router
            Status: In Progress

# Agent: Router
## Task: Analyse the keywords in the question Tell me about self-attention mechanism in Transformers?Based on the keywords decide whether it is eligible for a vectorstore search or a web search.Return a single word 'vectorstore' if it is eligible for vectorstore search.Return a single word 'websearch' if it is eligible for web search.Do not provide any other premable or explaination.


🤖 Agent: Router
    Status: In Progress
└── 🧠 Thinking...

...

# Agent: Answer Grader
## Task: Based on the response from the hallucination task for the quetion Tell me about self-attention mechanism in Transformers? evaluate whether the answer is useful to resolve the question.If the answer is 'yes' return a clear and concise answer.If the answer is 'no' then perform a 'websearch' and return the response


# Agent: Answer Grader
## Final Answer: 
The self-attention mechanism in Transformers allows the model to weigh the importance of different words in a sentence when encoding them. It computes a set of attention scores that determine how much focus to place on other words in the input sequence for each word being processed. This is achieved through three main components: queries, keys, and values. Each word is transformed into these three vectors, and the attention scores are calculated by taking the dot product of the query with all keys, followed by a softmax operation to normalize the scores. The output is a weighted sum of the values, allowi

🚀 Crew: crew
├── 📋 Task: 6cb1dd88-792a-435a-84a6-58906208a4dc
│      Assigned to: Router
│      Status: ✅ Completed
│   └── 🤖 Agent: Router
│           Status: ✅ Completed
├── 📋 Task: 26d34c08-b875-42ee-902c-b9ba4dc823c2
│      Assigned to: Retriever
│      Status: ✅ Completed
│   └── 🤖 Agent: Retriever
│           Status: ✅ Completed
├── 📋 Task: e07bb0f9-4e81-4830-b34e-db9cdceb6e30
│      Assigned to: Answer Grader
│      Status: ✅ Completed
│   └── 🤖 Agent: Answer Grader
│           Status: ✅ Completed
├── 📋 Task: a7bb2581-a3c3-4130-bbb5-9a650d0709fd
│      Assigned to: Hallucination Grader
│      Status: ✅ Completed
│   └── 🤖 Agent: Hallucination Grader
│           Status: ✅ Completed
└── 📋 Task: 6545f4ea-e2e2-4f3c-87e5-634b57ac9346
       Status: Executing Task...
    └── 🤖 Agent: Answer Grader
            Status: ✅ Completed

🚀 Crew: crew
├── 📋 Task: 6cb1dd88-792a-435a-84a6-58906208a4dc
│      Assigned to: Router
│      Status: ✅ Completed
│   └── 🤖 Agent: Router
│           Status: ✅ Completed
├── 📋 Task: 26d34c08-b875-42ee-902c-b9ba4dc823c2
│      Assigned to: Retriever
│      Status: ✅ Completed
│   └── 🤖 Agent: Retriever
│           Status: ✅ Completed
├── 📋 Task: e07bb0f9-4e81-4830-b34e-db9cdceb6e30
│      Assigned to: Answer Grader
│      Status: ✅ Completed
│   └── 🤖 Agent: Answer Grader
│           Status: ✅ Completed
├── 📋 Task: a7bb2581-a3c3-4130-bbb5-9a650d0709fd
│      Assigned to: Hallucination Grader
│      Status: ✅ Completed
│   └── 🤖 Agent: Hallucination Grader
│           Status: ✅ Completed
└── 📋 Task: 6545f4ea-e2e2-4f3c-87e5-634b57ac9346
       Assigned to: Answer Grader
       Status: ✅ Completed
    └── 🤖 Agent: Answer Grader
            Status: ✅ Completed

╭──────────────────────────────────────────────── Task Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Task Completed                                                                                                 │
│  Name: 6545f4ea-e2e2-4f3c-87e5-634b57ac9346                                                                     │
│  Agent: Answer Grader                                                                                           │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

╭──────────────────────────────────────────────── Crew Completion ────────────────────────────────────────────────╮
│                                                                                                                 │
│  Crew Execution Completed                                                                                       │
│  Name: crew                                                                                                     │
│  ID: 14fb2359-4057-4646-b909-5000043623b0                                                                       │
│                                                                                                                 │
│                                                                                                                 │
╰─────────────────────────────────────────────────────────────────────────────────────────────────────────────────╯

In [28]:
print(result)

The self-attention mechanism in Transformers allows the model to weigh the importance of different words in a sentence when encoding them. It computes a set of attention scores that determine how much focus to place on other words in the input sequence for each word being processed. This is achieved through three main components: queries, keys, and values. Each word is transformed into these three vectors, and the attention scores are calculated by taking the dot product of the query with all keys, followed by a softmax operation to normalize the scores. The output is a weighted sum of the values, allowing the model to capture contextual relationships and dependencies between words effectively. This mechanism is crucial for tasks such as translation, summarization, and other natural language processing applications.
